# Test Type A temporal samples

Checks the complete NPZ -> multiscale graph -> six-step temporal-sample
workflow end to end.

Each sample uses `[h(t-2), h(t-1), h(t), P(t)]` and stores targets
`[h(t+1), ..., h(t+6)]`.

### Imports

In [1]:
from pathlib import Path
import sys

import torch
from torch_geometric.loader import DataLoader

PROJECT_ROOT = Path(r"C:\Users\Prof. KN Jha\Desktop\SFINCS_mSWE-GNN_v3")

DATASET_SPLIT_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "batch_type_A_training_npz"      # Fixed: split file lives inside this folder, not directly under outputs/.
    / "dataset_split_type_A.json"
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Project root exists:", PROJECT_ROOT.exists())
print("Dataset split:", DATASET_SPLIT_PATH)
print("Dataset split exists:", DATASET_SPLIT_PATH.exists())

c:\Users\Prof. KN Jha\miniconda3\envs\hydromt-sfincs\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: C:\Users\Prof. KN Jha\Desktop\SFINCS_mSWE-GNN_v3
Project root exists: True
Dataset split: C:\Users\Prof. KN Jha\Desktop\SFINCS_mSWE-GNN_v3\outputs\batch_type_A_training_npz\dataset_split_type_A.json
Dataset split exists: True


### Project function imports 

In [2]:
from scripts.sfincs_dataset_conversion.pytorch_geometric import (
    convert_one_npz_to_graph,
)

from scripts.sfincs_dataset_conversion.temporal_learning import (
    DEFAULT_PREDICTION_HORIZON,
    convert_one_graph_to_samples,
)

from scripts.sfincs_dataset_conversion.type_a_data import (
    read_dataset_split_file,
)

## Build one complete graph

In [3]:
dataset_split = read_dataset_split_file(DATASET_SPLIT_PATH)

npz_path = Path(dataset_split["train"][0])

if not npz_path.is_absolute():
    npz_path = PROJECT_ROOT / npz_path

graph = convert_one_npz_to_graph(npz_path)

print("NPZ file:", npz_path.name)
print("Case:", graph.case_name)
print(
    "Graph hierarchy:",
    f"{graph.fine_ny} x {graph.fine_nx}",
    "->",
    f"{graph.medium_ny} x {graph.medium_nx}",
    "->",
    f"{graph.coarse_ny} x {graph.coarse_nx}",
)
print("Water depth:", tuple(graph.water_depth.shape))
print("Rainfall:", tuple(graph.rainfall.shape))
print("Model times:", tuple(graph.model_time_seconds.shape))

NPZ file: DEM_41_L3_RP002_temporal_uniform_type_A.npz
Case: DEM_41_L3_RP002_temporal_uniform_type_A
Graph hierarchy: 64 x 64 -> 32 x 32 -> 16 x 16
Water depth: (4096, 91)
Rainfall: (4096, 91)
Model times: (91,)


## Validate the complete graph

In [4]:
number_of_nodes = int(graph.num_fine_nodes)
number_of_frames = graph.water_depth.shape[1]

assert graph.water_depth.shape == (number_of_nodes, number_of_frames)
assert graph.rainfall.shape == (number_of_nodes, number_of_frames)
assert graph.model_time_seconds.shape == (number_of_frames,)
assert torch.all(graph.model_time_seconds[1:] > graph.model_time_seconds[:-1])

print("Complete graph shapes passed.")

Complete graph shapes passed.


## Build temporal samples

In [5]:
temporal_samples = convert_one_graph_to_samples(graph)
sample = temporal_samples[0]  # Starts at t = 2, the earliest valid index.

expected_samples = number_of_frames - DEFAULT_PREDICTION_HORIZON - 2
assert len(temporal_samples) == expected_samples

assert sample.fine_dynamic_x.shape == (number_of_nodes, 4)
assert sample.rainfall_sequence.shape == (number_of_nodes, DEFAULT_PREDICTION_HORIZON)
assert sample.y.shape == (number_of_nodes, DEFAULT_PREDICTION_HORIZON)
assert sample.delta_time_seconds.shape == (1, DEFAULT_PREDICTION_HORIZON)

print("Water-depth frames:", number_of_frames)
print("Temporal samples:", len(temporal_samples))
print("Dynamic input:", tuple(sample.fine_dynamic_x.shape))
print("Rainfall sequence:", tuple(sample.rainfall_sequence.shape))
print("Target sequence:", tuple(sample.y.shape))
print("Delta-time sequence:", tuple(sample.delta_time_seconds.shape))

Water-depth frames: 91
Temporal samples: 83
Dynamic input: (4096, 4)
Rainfall sequence: (4096, 6)
Target sequence: (4096, 6)
Delta-time sequence: (1, 6)


## Verify temporal indexing

In [ ]:
assert int(sample.time_index) == 2

# torch.equal() checks if two tensors have the same size and elements.
assert torch.equal(sample.fine_dynamic_x[:, 0:1], graph.water_depth[:, 0:1])  # h(t-2)
assert torch.equal(sample.fine_dynamic_x[:, 1:2], graph.water_depth[:, 1:2])  # h(t-1)
assert torch.equal(sample.fine_dynamic_x[:, 2:3], graph.water_depth[:, 2:3])  # h(t)
assert torch.equal(sample.fine_dynamic_x[:, 3:4], graph.rainfall[:, 2:3])     # P(t)

assert torch.equal(sample.rainfall_sequence, graph.rainfall[:, 2:8])         # P(t) to P(t+5)
assert torch.equal(sample.y, graph.water_depth[:, 3:9])                      # h(t+1) to h(t+6)

expected_delta_time = (
    graph.model_time_seconds[3:9] - graph.model_time_seconds[2:8]
).unsqueeze(0)
assert torch.equal(sample.delta_time_seconds, expected_delta_time)

print("Temporal indexing passed.")

Temporal indexing passed.


## Test PyG Batching

In [7]:
data_loader = DataLoader(temporal_samples[:2], batch_size=2, shuffle=False)
batch = next(iter(data_loader))

assert batch.num_graphs == 2
assert batch.fine_dynamic_x.shape == (2 * number_of_nodes, 4)
assert batch.rainfall_sequence.shape == (2 * number_of_nodes, DEFAULT_PREDICTION_HORIZON)
assert batch.y.shape == (2 * number_of_nodes, DEFAULT_PREDICTION_HORIZON)
assert batch.delta_time_seconds.shape == (2, DEFAULT_PREDICTION_HORIZON)

print("Batched dynamic input:", tuple(batch.fine_dynamic_x.shape))
print("Batched rainfall sequence:", tuple(batch.rainfall_sequence.shape))
print("Batched target sequence:", tuple(batch.y.shape))
print("Batched delta time:", tuple(batch.delta_time_seconds.shape))

Batched dynamic input: (8192, 4)
Batched rainfall sequence: (8192, 6)
Batched target sequence: (8192, 6)
Batched delta time: (2, 6)


## Verify multiscale index offsets after batching

Confirms PyG correctly shifted the second graph's fine/medium/coarse edge
indices and pooling maps so they don't collide with the first graph's
node numbering.

In [8]:
index_checks = [
    ("Fine edges", batch.fine_edge_index, int(sample.num_fine_nodes), batch.fine_z.shape[0]),
    ("Medium edges", batch.medium_edge_index, int(sample.num_medium_nodes), batch.medium_z.shape[0]),
    ("Coarse edges", batch.coarse_edge_index, int(sample.num_coarse_nodes), batch.coarse_z.shape[0]),
    ("Fine-to-medium map", batch.fine_to_medium, int(sample.num_medium_nodes), batch.medium_z.shape[0]),
    ("Medium-to-coarse map", batch.medium_to_coarse, int(sample.num_coarse_nodes), batch.coarse_z.shape[0]),
]

for check_name, indices, second_graph_start, total_nodes in index_checks:
    assert int(indices.max()) >= second_graph_start
    assert int(indices.max()) < total_nodes
    print(check_name, "passed.")

print("PyG batching passed.")

Fine edges passed.
Medium edges passed.
Coarse edges passed.
Fine-to-medium map passed.
Medium-to-coarse map passed.
PyG batching passed.


## Summary

In [9]:
print("TYPE A TEMPORAL SAMPLE TEST PASSED")
print()
print("Case:", graph.case_name)
print(
    "Hierarchy:",
    f"{graph.fine_ny} x {graph.fine_nx}",
    "->",
    f"{graph.medium_ny} x {graph.medium_nx}",
    "->",
    f"{graph.coarse_ny} x {graph.coarse_nx}",
)
print("Temporal samples:", len(temporal_samples))
print("Input: [h(t-2), h(t-1), h(t), P(t)]")
print("Targets: [h(t+1), ..., h(t+6)]")
print("PyG batching: passed")

TYPE A TEMPORAL SAMPLE TEST PASSED

Case: DEM_41_L3_RP002_temporal_uniform_type_A
Hierarchy: 64 x 64 -> 32 x 32 -> 16 x 16
Temporal samples: 83
Input: [h(t-2), h(t-1), h(t), P(t)]
Targets: [h(t+1), ..., h(t+6)]
PyG batching: passed
